# Customer Churn Over Time

In this notebook, we focused on determining **customer churn status** on a **monthly** basis using a series of data transformation steps. **Customer churn**, in this context, is defined as a time-series event that helps us identify when a customer has effectively stopped engaging with the café’s business. Churn is calculated by first determining the interval days between purchase orders for each customer, followed by calculating the standard deviation of those intervals. The churn threshold for each customer is then set as twice their standard deviation, which was calculated in another notebook named **Cafe_Analytics_Data_Cleaning_MX.ipynb**.

The churn is calculated on a monthly basis to provide a more granular understanding of customer retention trends over time. By analyzing churn monthly, we can capture any seasonal or short-term fluctuations in customer behavior and assess their impact on the business. This approach helps the café respond more promptly to changes in customer activity and design targeted strategies to retain customers at different points throughout the year. 

In this notebook, we applied the churn threshold to each customer’s purchase history. Specifically, we computed the interval days between each month-end and the customer's most recent purchase before that month-end. If the interval exceeds the customer's churn threshold, the customer is marked as **CHURN**; otherwise they are marked as **NOT CHURN**. This allows for a consistent, time-bound assessment of customer engagement.

The processing steps to determine customer churn status for each customer on a monthly basis include:
1) [Load and Preprocess Data to include Appropriate Columns](#load_process)
2) [Compute the Month-end Date for Each Month of Year](#month-end)
3) [Merge Month-end Dates into the Churn Table](#merge)
4) [Identify the Latest Purchase Time for Each Customer Per Month](#latest-purchase)
5) [Calculate the Interval Days between each Month-end and corresponding Latest Purchase Time](#interval)
6) [Use the Interval Days and Churn Threshold to determine each customer's churn status for every month](#churn-label)
7) [Export to Parquet Files](#parquet)

In [ ]:
import pandas as pd
import os
import json
import numpy as np
from datetime import datetime, date, timedelta

In [ ]:
# Set the maximum column width for pandas DataFrame display
pd.set_option('max_colwidth', 2000)

<a name="load_process"></a>
### Load and Preprocess Data

In [ ]:
# Load the CSV file into a pandas DataFrame and 
# parse the 'order_time' column as datetime
df = pd.read_csv(
    './cafe_analytics_cleaned_CSV_MX/cafe_analytics_full_mx.csv',
    parse_dates=['order_time'],
)

In [ ]:
# Extract necessary columns and remove duplicates
df_custchurn = df[
    ['order_time', 'customer_id', 'churn_threshold']
].drop_duplicates().copy()

In [ ]:
# Print the shape and information of the extracted DataFrame
print(df_custchurn.shape)
print(df_custchurn.info())

In [ ]:
df_custchurn.head()

<a name="month-end"></a>
### Compute the Month-end Date for Each Month of Year

In [ ]:
# Get the earliest order date
start_date = df_custchurn['order_time'].min().date()
# Get the latest order date
end_date = df_custchurn['order_time'].max().date()
# Create a new DataFrame with a range of dates from 
# start_date to end_date
df_dates = pd.DataFrame(
    {"Dates": pd.date_range(start_date, end_date)}
)

In [ ]:
df_dates.head()

In [ ]:
# Extract year and month from 'Dates' column
df_dates['year'] = df_dates['Dates'].dt.year
df_dates['month'] = df_dates['Dates'].dt.month

In [ ]:
df_dates.head()

In [ ]:
# Get the latest purchase date
latest_date = df_dates['Dates'].max()

def calc_monthend(x):
    """ A function to calculate the begin date of next month (month-end) based 
    on the given year and month.
    """
    # Check if the current year and month are the same as 
    # the latest date's year and month
    if (x['year'] == latest_date.year) and (x['month'] == latest_date.month):
        # If true, return the day after the latest date
        return latest_date + timedelta(days=1)
    else:
        # Check if the month is less than December
        if x['month'] < 12:
            # Return the 1st day of the next month
            return date(x['year'], x['month'] + 1, 1)
        else:
            # If it's December, return the 1st day of Jan next year 
            return date(x['year'] + 1, 1, 1)

# Apply the calc_monthend function to each row in df_dates to 
# calculate the month-end
df_dates['month_end'] = df_dates.apply(calc_monthend, axis=1)

In [ ]:
# Select only the 'year', 'month', and 'month_end' columns from df_dates
# and remove duplicate rows to ensure each year and month combination is unique
df_dates = df_dates[['year', 'month', 'month_end']].drop_duplicates()
df_dates

<a name="merge"></a>
### Merge Month-end Dates into the Churn Table

In [ ]:
# Perform a cross join between df_custchurn and the dates DataFrame
df_churn = df_custchurn.merge(
    df_dates,
    how='cross',
)

In [ ]:
df_churn.head()

<a name="latest-purchase"></a>
### Identify the Latest Purchase Time for Each Customer Per Month

In [ ]:
# Filter the churn data for records where the order time 
# is earlier than the month-end
df_churn = df_churn[
    df_churn['order_time'] < df_churn['month_end']
]

In [ ]:
# Sort and reset the index for clean display
df_churn = df_churn[
    [
        'month_end', 'customer_id', 'order_time', 
        'year', 'month', 'churn_threshold'
    ]
].sort_values(
    ['month_end', 'customer_id', 'order_time'],
    ascending=[True, True, True],
).reset_index(drop=True)

df_churn.head(10)

In [ ]:
# Add a column for the latest order time per customer for each month
df_churn['latest_order_time'] = (
    df_churn.groupby(
        ['month_end', 'customer_id']
    )
    ['order_time'].transform(np.max)
)

df_churn.head()

In [ ]:
# Drop the original 'order_time' column and remove duplicates
df_churn = df_churn.drop(
    'order_time', axis=1
).drop_duplicates()

In [ ]:
df_churn.head(10)

<a name="interval"></a>
### Calculate Interval Days

Compute the interval days between each month-end and the customer's most recent purchase before that month-end.

In [ ]:
# Define a function to compute the number of days between 
# the latest purchase and the month-end
def compute_interval(x):
    return (
        x['month_end'].date() - x['latest_order_time'].date()
    ).days
    
# Apply the function to compute the interval between the 
# latest purchase and the month-end
df_churn['interval_to_latest_purchase'] = (
    df_churn.apply(compute_interval, axis=1)
)

In [ ]:
df_churn.head(10)

<a name="churn-label"></a>
### Determine Customer Churn Status over Time

A customer will be classified as **CHURN** if the interval days exceed their churn threshold; otherwise, they will be identified as **NOT CHURN**.

In [ ]:
# Determine churn status by checking if the interval 
# exceeds the churn threshold
df_churn['churn'] = (
    df_churn['interval_to_latest_purchase']
    > df_churn['churn_threshold']
)

In [ ]:
# Display churn information for the first 20 customers on the latest date
df_churn[
    df_churn['month_end'] == latest_date + timedelta(days=1)
].head(20)

The customers marked as **True** in the `churn` column have stopped engaging with the café’s business, while those marked as **False** are still active customers.

In [ ]:
# Final clean-up: drop 'month_end', 
# sort by customer ID, year, and month, and reset index
df_churn_final = (
    df_churn.drop('month_end', axis=1)
    .sort_values(
        ['customer_id', 'year', 'month'],
        ascending=[True, True, True],
    ).reset_index(drop=True)
)

In [ ]:
# Display the data for customer with ID 5
df_churn_final[
    df_churn_final['customer_id'] == 5
]

<a name="parquet"></a>
### Export to Parquet Files

In [ ]:
# Create the output folder if it doesn't exist
output_folder = 'cafe_processed_files'
if not os.path.exists(os.path.join(os.getcwd(), output_folder)):
    os.mkdir(output_folder)

# Save the final churn data to a PARQUET file
df_churn_final.to_parquet(
    os.path.join(output_folder, 'cafe_customer_churn_mx.parquet'),
)